# Модуль-ноутбук: `labels`

Мітка C (within-creator engagement). Пороги медіан рахуються **лише на train** і заморожуються. Єдине місце, де читаються post-hoc метрики — і лише щоб ВИЗНАЧИТИ ціль.

**Залежності:** `%run` 00_config.ipynb

In [ ]:
%run 00_config.ipynb

In [ ]:
"""Label C: within-creator relative engagement (see PLAN.md §2).

These functions are the ONLY place post-hoc metrics are read, and only to DEFINE the
target. The thresholds are fit on the training split and frozen; the same frozen
thresholds label the test split. The label is never fed back as a feature.
"""

import numpy as np
import pandas as pd

In [ ]:
def compute_engagement_rate(df: pd.DataFrame) -> pd.Series:
    """ER = (digg+share+comment+collect) / play_count. Post-hoc only. NaN if play<=0."""
    play = pd.to_numeric(df["play_count"], errors="coerce")
    eng = sum(pd.to_numeric(df[c], errors="coerce").fillna(0) for c in config.ENGAGEMENT_COLS)
    er = eng / play.where(play > 0)
    return er.astype(float)

In [ ]:
def fit_creator_thresholds(train_df: pd.DataFrame) -> dict:
    """Per-creator median ER + global median fallback — FIT ON TRAIN ROWS ONLY."""
    er = compute_engagement_rate(train_df)
    tmp = pd.DataFrame({"creator": train_df[config.CREATOR_COL].values, "er": er.values})
    tmp = tmp.dropna(subset=["er"])
    per_creator = tmp.groupby("creator")["er"].median().to_dict()
    global_median = float(tmp["er"].median())
    return {"per_creator": per_creator, "global_median": global_median}

In [ ]:
def make_labels(df: pd.DataFrame, thresholds: dict) -> pd.Series:
    """1 if a video's ER strictly exceeds its creator's (train) median ER, else 0.

    Unseen creator -> global train median. Rows with undefined ER -> NaN (dropped upstream).
    """
    er = compute_engagement_rate(df)
    per_creator = thresholds["per_creator"]
    g = thresholds["global_median"]
    ref = df[config.CREATOR_COL].map(lambda c: per_creator.get(c, g)).astype(float)
    label = (er > ref).astype(float)
    label[er.isna()] = np.nan
    return label.rename("label")

In [ ]:
from types import SimpleNamespace
labels = SimpleNamespace(
    compute_engagement_rate=compute_engagement_rate,
    fit_creator_thresholds=fit_creator_thresholds,
    make_labels=make_labels,
)